# Fine-tune Qwen2.5-3B-Instruct bằng Unsloth (DoRA) trên VlogQA
Notebook này hướng dẫn fine-tune mô hình **Qwen2.5-3B-Instruct** với **DoRA** qua thư viện **Unsloth** cho tập dữ liệu **VlogQA** (đọc hiểu hội thoại tiếng Việt).

VlogQA là dataset **Extractive QA** (dạng SQuAD): mô hình cần trích xuất một đoạn văn bản ngắn từ context làm câu trả lời.

**Evaluation metrics:** Exact Match (EM) và F1-score.

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ để tránh xung đột
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [1]:
# Bước 2: Cài đặt lại PyTorch với CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 10.9 MB/s  0:01:00:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 11.0 MB/s  0:00:26:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 10.8 MB/s  0:00:27:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 10.9 MB/s  0:00:12:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 10.9 MB/s  0:00:55:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 11.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 10.8 MB/s  0:00:17:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 9.1 MB/s  0:00:01eta 0:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 9.2 MB/s  0:00:07 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 

In [2]:
# Bước 3: Cài đặt transformers, trl, peft, accelerate, bitsandbytes, xformers và unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 953.2 kB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 1.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 1.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 2.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 2.2 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 2.3 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 2.6 MB/s  0:00:23m0:00:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 5.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 3.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 5.6 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/8

## 1. Load Model và Tokenizer với Unsloth

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192  # Tăng từ 1536 → bao phủ P90 VlogQA thực tế (~7052 tokens)
dtype = None           # None để auto-detect (BFloat16 cho 5070 Ti - Blackwell arch)
load_in_4bit = False   # LoRA thuần: nạp model BF16 (~6GB), không quantize

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Tải model thành công!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/venv/lib/python3.12/site-packages/unsloth_zoo/_vendored/fla/utils/_device.py:100: UserWarning: Triton is not supported on current platform, roll back to CPU.
  _cpu_device_warning()
/opt/venv/lib/python3.12/site-packages/unsloth_zoo/_vendored/fla/utils/_device.py:165: UserWarning: Triton is not supported on current platform, roll back to CPU.
  _cpu_device_warning()
/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behav

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.682 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Tải model thành công!


## 2. Cấu hình LoRA Adapter

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # Rank của LoRA (16 là cân bằng tốt)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,                 # Scaling factor (thường = 2 * r)
    lora_dropout = 0.05,             # Dropout nhẹ để tránh overfitting
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Tiết kiệm VRAM 30%
    random_state = 3407,
    use_dora  = True,            # ← DoRA: Weight-Decomposed LoRA
    use_rslora = False,
    loftq_config = None,
)

print("Cấu hình LoRA thành công!")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Cấu hình LoRA thành công!


## 3. Chuẩn bị Dataset VlogQA

In [3]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                # Lấy câu trả lời đầu tiên (có thể có nhiều annotators)
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""  # Bỏ qua câu không có đáp án
                if answer:  # Chỉ lấy mẫu có đáp án
                    samples.append({
                        "context": context,
                        "question": question,
                        "answer": answer,
                        "id": qa.get("id", ""),
                    })
    return samples

# Đường dẫn dữ liệu (điều chỉnh nếu cần)
TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"
TEST_PATH  = "test.json"

train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")
print(f"Ví dụ mẫu đầu tiên:")
print(f"  Context (100 ký tự đầu): {train_samples[0]['context'][:100]}")
print(f"  Question: {train_samples[0]['question']}")
print(f"  Answer: {train_samples[0]['answer']}")

Tổng số mẫu train: 8386
Ví dụ mẫu đầu tiên:
  Context (100 ký tự đầu): a cho cô bán anh chị em và các bạn Hôm nay mình khuya sẽ chia sẻ món sườn non sốt bơ tỏi Ừ để mình c
  Question: Nước tương được sử dụng để nêm hãng gì?
  Answer: Maggi


In [4]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""
                if answer:
                    samples.append({
                        "context":  context,
                        "question": question,
                        "answer":   answer,
                        "id":       qa.get("id", ""),
                    })
    return samples


# ==========================================
# Anchor-based Context Truncation
# ==========================================
MAX_CONTEXT_TOKENS = 7500

def truncate_context_around_answer(context, answer, tokenizer, max_ctx_tokens=MAX_CONTEXT_TOKENS):
    answer_start = context.find(answer)
    if answer_start == -1:
        ctx_ids = tokenizer.encode(context, add_special_tokens=False)
        if len(ctx_ids) <= max_ctx_tokens:
            return context
        return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)

    before_text = context[:answer_start]
    after_text  = context[answer_start + len(answer):]

    before_ids = tokenizer.encode(before_text, add_special_tokens=False)
    answer_ids = tokenizer.encode(answer,       add_special_tokens=False)
    after_ids  = tokenizer.encode(after_text,   add_special_tokens=False)

    if len(before_ids) + len(answer_ids) + len(after_ids) <= max_ctx_tokens:
        return context

    budget = max_ctx_tokens - len(answer_ids)
    half   = budget // 2

    before_keep = before_ids[-half:]
    after_keep  = after_ids[:budget - half]

    merged_ids = before_keep + answer_ids + after_keep
    return tokenizer.decode(merged_ids, skip_special_tokens=True)


# ==========================================
# System Prompt & User Template (đồng bộ Train/Eval)
# ==========================================
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện nguyên văn trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm \'Đáp án:\' hay bất kỳ tiền tố nào.\n"
    "3) Câu trả lời phải là một chuỗi ký tự CÓ MẶT trong đoạn văn được cung cấp."
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Câu trả lời (span-only):"
)


def format_prompt_train(context, question, answer, tokenizer):
    """Tạo prompt đầy đủ (input + output) để huấn luyện."""
    context_cropped = truncate_context_around_answer(context, answer, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context_cropped,
                question=question
            )
        },
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def format_prompt_inference(context, question, tokenizer):
    """Tạo prompt inference (chỉ input, không có answer)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context[:7500],
                question=question
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [10]:
# ==========================================
# TẠO DATASET CHO HUẤN LUYỆN VÀ VALIDATION
# ==========================================
from datasets import Dataset

TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"

# --- Train dataset ---
train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")

print("\nĐang format prompt cho train dataset...")
train_texts = [
    format_prompt_train(
        context   = s["context"],
        question  = s["question"],
        answer    = s["answer"],
        tokenizer = tokenizer
    )
    for s in train_samples
]
dataset = Dataset.from_dict({"text": train_texts})
print(f"Train dataset: {len(dataset)} mẫu")

# --- Eval dataset (dùng dev.json cho Early Stopping) ---
dev_samples = load_vlogqa(DEV_PATH)
print(f"\nTổng số mẫu dev: {len(dev_samples)}")

print("Đang format prompt cho eval dataset...")
dev_texts = [
    format_prompt_train(
        context   = s["context"],
        question  = s["question"],
        answer    = s["answer"],
        tokenizer = tokenizer
    )
    for s in dev_samples
]
eval_dataset = Dataset.from_dict({"text": dev_texts})
print(f"Eval dataset: {len(eval_dataset)} mẫu")

print(f"\nVí dụ prompt train (100 ký tự đầu):")
print(f"{train_texts[0][:150]}...")

Tổng số mẫu train: 8386

Đang format prompt cho train dataset...
Train dataset: 8386 mẫu

Tổng số mẫu dev: 999
Đang format prompt cho eval dataset...
Eval dataset: 999 mẫu

Ví dụ prompt train (100 ký tự đầu):
<|im_start|>system
Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. QUY TẮC BẮT BUỘC:
1) Chỉ trả về ĐÚNG cụm từ xuất hiện nguyên văn tron...


## 4. Cấu hình và Bắt đầu Huấn luyện (Fine-Tuning)

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported
import sys

# Số epoch tối đa (Early Stopping sẽ dừng sớm hơn nếu cần)
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2   # Dừng nếu eval_loss không giảm sau 2 epoch liên tiếp

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    eval_dataset     = eval_dataset,   # ← Thêm validation set để Early Stopping hoạt động
    dataset_text_field = "text",
    max_seq_length   = max_seq_length,
    dataset_num_proc = 1,
    packing          = True,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 8,       # Effective batch = 16
        warmup_steps                 = 50,
        num_train_epochs             = MAX_EPOCHS,
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 20,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",
        seed                         = 3407,
        output_dir                   = "outputs_vlogqa_dora",

        # === Cấu hình Evaluation & Early Stopping ===
        eval_strategy                = "epoch",     # Đánh giá sau mỗi epoch
        save_strategy                = "epoch",     # Lưu checkpoint sau mỗi epoch
        load_best_model_at_end       = True,        # Tự động load checkpoint tốt nhất khi kết thúc
        metric_for_best_model        = "eval_loss", # Dùng eval_loss làm tiêu chí
        greater_is_better            = False,       # eval_loss càng nhỏ càng tốt
        save_total_limit             = 2,           # Giữ tối đa 2 checkpoint (tiết kiệm ổ cứng)

        report_to                    = "none",
    ),
)

# Fix sys.modules để tránh lỗi pickle với unsloth
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

# Bắt đầu train!
print(f"\nBắt đầu fine-tuning với Early Stopping (patience={EARLY_STOPPING_PATIENCE}, max {MAX_EPOCHS} epochs)...")
trainer_stats = trainer.train()

# In tóm tắt kết quả
print(f"\nHoàn tất Training!")
print(f"Epochs đã chạy: {trainer_stats.metrics.get('epoch', 'N/A'):.2f}")
print(f"Train Loss cuối: {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")

DORA_SAVE_PATH = "qwen2.5-3b-instruct-dora-vlogqa"

model.save_pretrained(DORA_SAVE_PATH)
tokenizer.save_pretrained(DORA_SAVE_PATH)

print(f"Đã lưu thành công mô hình LoRA tại: {DORA_SAVE_PATH}")


Bắt đầu fine-tuning với Early Stopping (patience=2, max 5 epochs)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,386 | Num Epochs = 5 | Total steps = 2,625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Epoch,Training Loss,Validation Loss
1,1.442025,3.080135
2,0.377327,4.002535
3,0.087939,5.159134


/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `


Hoàn tất Training!
Epochs đã chạy: 3.00
Train Loss cuối: 1.0251


Unsloth: Restored added_tokens_decoder metadata in qwen2.5-3b-instruct-lora-vlogqa/tokenizer_config.json.


Đã lưu thành công mô hình LoRA tại: qwen2.5-3b-instruct-lora-vlogqa


## 5. Kiểm tra Inference nhanh sau khi huấn luyện

In [13]:
# Kích hoạt chế độ sinh token nhanh của Unsloth (2x tốc độ)
# DoRA: Tắt FastInference, dùng model.eval() + use_cache=False để tránh bug
model.eval()

# Thử nghiệm với mẫu đầu tiên trong tập train
sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 64,
    use_cache = False,  # DoRA yêu cầu tắt KV Cache khi chưa merge
    do_sample = False,    # Greedy decoding cho QA
    temperature = 1.0,
)

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE ---")
print(f"Câu hỏi : {sample['question']}")
print(f"Đáp án đúng : {sample['answer']}")
print(f"Model trả lời: {response}")

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- KẾT QUẢ INFERENCE ---
Câu hỏi : Nước tương được sử dụng để nêm hãng gì?
Đáp án đúng : Maggi
Model trả lời: hoa ốc ốc bò


## 6. Lưu Model LoRA

In [ ]:
# Chỉ lưu LoRA weights (nhẹ, vài chục MB)
DORA_SAVE_PATH = "qwen2.5-3b-instruct-dora-vlogqa"

model.save_pretrained(DORA_SAVE_PATH)
tokenizer.save_pretrained(DORA_SAVE_PATH)

print(f"Đã lưu thành công mô hình LoRA tại: {DORA_SAVE_PATH}")

# Tùy chọn: Merge LoRA vào model gốc và lưu dạng full weights
# model.save_pretrained_merged("qwen2.5-1.5b-instruct-vlogqa-merged", tokenizer, save_method="merged_16bit")

---
## 7. Đánh giá (Evaluation) trên tập Test
### Metrics: Exact Match (EM) và F1-score

- **Exact Match (EM):** Tỷ lệ câu trả lời của model khớp **hoàn toàn** với đáp án (sau khi chuẩn hóa text).
- **F1-score:** Đo lường overlap từ-vựng giữa dự đoán và đáp án (tính theo cấp độ token).

> **Lưu ý:** Cell này có thể chạy trong một session mới sau khi đã train và lưu model.

In [3]:
import json
import re
import string
import unicodedata
import torch
from tqdm.auto import tqdm
from collections import Counter

# ==========================================
# CẤU HÌNH
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH    = "qwen2.5-3b-instruct-dora-vlogqa"
TEST_DATA_PATH  = "test.json"
MAX_EVAL_TOKENS = 7500

# ==========================================
# ĐỒNG BỘ PROMPT (Giống hệt lúc train)
# ==========================================
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện nguyên văn trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm \'Đáp án:\' hay bất kỳ tiền tố nào.\n"
    "3) Câu trả lời phải là một chuỗi ký tự CÓ MẶT trong đoạn văn được cung cấp."
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Câu trả lời (span-only):"
)

# Regex dọn sạch tiền tố thừa mà model hay sinh ra
PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn|trả lời|span-only)\s*[:\-]?\s*",
    re.IGNORECASE,
)


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# ==========================================
# LOAD MODEL ĐÃ FINE-TUNE (DoRA → Merge → Fast Batch Inference)
# ==========================================
from unsloth import FastLanguageModel
from peft import PeftModel
import torch
import os

# Load từ local cache nếu HuggingFace timeout
os.environ["HF_HUB_OFFLINE"] = "1"

print("Bước 1: Load Base Model...")
# Load base model trước (không kèm adapter)
BASE_MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
model_eval, tokenizer_eval = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_NAME,
    max_seq_length = 8192,
    dtype          = torch.bfloat16,
    load_in_4bit   = False,
)
print(f"  Base model loaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

print("Bước 2: Gắn DoRA Adapter...")
model_eval = PeftModel.from_pretrained(
    model_eval,
    ADAPTER_PATH,
    is_trainable = False,
)
print(f"  Adapter gắn xong. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

print("Bước 3: Merge Adapter vào Base Model (mất ~10-30s)...")
model_eval = model_eval.merge_and_unload()
model_eval.eval()
print(f"  Merge hoàn tất! VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Bắt buộc padding BÊN TRÁI cho batch generation
tokenizer_eval.padding_side = "left"
if tokenizer_eval.pad_token is None:
    tokenizer_eval.pad_token = tokenizer_eval.eos_token

# Tắt warning max_length vs max_new_tokens
if hasattr(model_eval, "generation_config"):
    model_eval.generation_config.max_length = None

print("\nModel sẵn sàng Batch Inference với use_cache=True (model thuần, không còn adapter)!")


Bước 1: Load Base Model...
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.684 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


  Base model loaded. VRAM: 12.60 GB
Bước 2: Gắn DoRA Adapter...
  Adapter gắn xong. VRAM: 12.72 GB
Bước 3: Merge Adapter vào Base Model (mất ~10-30s)...
  Merge hoàn tất! VRAM: 12.60 GB

Model sẵn sàng Batch Inference với use_cache=True (model thuần, không còn adapter)!


In [5]:
# ==========================================
# HÀM TẠO PROMPT INFERENCE (đồng bộ với train)
# ==========================================
def format_prompt_inference_eval(context, question, tokenizer, max_tokens=7500):
    """Tạo prompt cho inference - cùng template với lúc train."""
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    if len(ctx_ids) > max_tokens:
        context = tokenizer.decode(ctx_ids[:max_tokens], skip_special_tokens=True)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context,
                question=question
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")


Tổng số câu hỏi test: 1062


In [6]:
# ==========================================
# HÀM TẠO PROMPT INFERENCE (đồng bộ với train)
# ==========================================
def format_prompt_inference_eval(context, question, tokenizer, max_tokens=7500):
    """Tạo prompt cho inference - cùng template với lúc train."""
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    if len(ctx_ids) > max_tokens:
        context = tokenizer.decode(ctx_ids[:max_tokens], skip_special_tokens=True)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context,
                question=question
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")


Tổng số câu hỏi test: 1062


In [7]:
# ==========================================
# HÀM TIỆN ÍCH: CHUẨN HÓA, METRIC & POST-PROCESSING
# ==========================================

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))


def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 and len(true_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)


def get_best_score(prediction: str, answers: list) -> tuple:
    """Tính EM và F1 tốt nhất so với tất cả các đáp án (best-of-N)."""
    em_scores = [compute_exact_match(prediction, ans["text"]) for ans in answers]
    f1_scores = [compute_f1_token(prediction, ans["text"]) for ans in answers]
    return max(em_scores), max(f1_scores)


def clean_prediction(raw: str) -> str:
    """Dọn sạch các tiền tố thừa mà model hay sinh ra (vd: 'Đáp án: ...')."""
    pred = raw.strip().split("\n")[0].strip().strip('"\'\' ')
    return PREFIX_RE.sub("", pred).strip()


def find_best_span(prediction: str, context: str) -> str:
    """
    Trượt sliding window qua context để tìm span có token-F1
    cao nhất so với prediction của model.
    Hiệu quả khi model paraphrase nhẹ thay vì copy nguyên văn.
    """
    pred_norm = normalize_text(prediction)
    if not pred_norm:
        return prediction

    pred_words = pred_norm.split()
    ctx_words  = context.split()
    n = len(pred_words)

    if n == 0 or len(ctx_words) == 0:
        return prediction

    best_f1   = compute_f1_token(prediction, context)
    best_span = prediction

    min_w = max(1, n // 2)
    max_w = min(2 * n + 5, len(ctx_words))

    for w in range(min_w, max_w + 1):
        for start in range(len(ctx_words) - w + 1):
            span_orig = " ".join(ctx_words[start:start + w])
            f1 = compute_f1_token(prediction, span_orig)
            if f1 > best_f1:
                best_f1   = f1
                best_span = span_orig

    # Chỉ trả về span mới nếu cải thiện thực sự
    return best_span if best_f1 >= 0.5 else prediction


print("Đã định nghĩa: normalize_text, compute_exact_match, compute_f1_token")
print("               get_best_score, clean_prediction, find_best_span")

Đã định nghĩa: normalize_text, compute_exact_match, compute_f1_token
               get_best_score, clean_prediction, find_best_span


In [13]:
# ==========================================
# CHẠY BATCH INFERENCE TRÊN TOÀN BỘ TẬP TEST
# ==========================================
import time
import warnings
warnings.filterwarnings("ignore", message="Both `max_new_tokens`")

# ---- Cấu hình Batch ----
# 3090 24GB: model đã merge ~6GB BF16, dư ~18GB → batch_size=8 thoải mái
BATCH_SIZE    = 8
SUMMARY_EVERY = 100

all_em_raw      = []
all_f1_raw      = []
all_em_span     = []
all_f1_span     = []
all_predictions = []

TOTAL      = len(test_samples)
n_batches  = (TOTAL + BATCH_SIZE - 1) // BATCH_SIZE
start_time = time.time()

print(f"\nBắt đầu đánh giá trên {TOTAL} câu hỏi...")
print(f"Batch size = {BATCH_SIZE} | Số batch = {n_batches} | use_cache=True (đã merge)\n")

pbar = tqdm(range(0, TOTAL, BATCH_SIZE), desc="Batch Evaluating", total=n_batches)

for batch_start in pbar:
    batch = test_samples[batch_start : batch_start + BATCH_SIZE]

    # Tạo prompt cho từng sample trong batch
    prompts = [
        format_prompt_inference_eval(
            context   = s["context"],
            question  = s["question"],
            tokenizer = tokenizer_eval
        )
        for s in batch
    ]

    # Tokenize cả batch (left-padding, cắt tối đa 8192)
    inputs = tokenizer_eval(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 8192,
    ).to(model_eval.device)

    total_input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model_eval.generate(
            **inputs,
            max_new_tokens = 64,
            do_sample      = False,
            temperature    = 1.0,
            use_cache      = True,  # An toàn sau merge - tăng tốc đáng kể
            pad_token_id   = tokenizer_eval.pad_token_id,
            eos_token_id   = tokenizer_eval.eos_token_id,
        )

    # Decode từng sample trong batch
    for j, sample in enumerate(batch):
        # Phần token mới sinh ra (sau phần input chung đã left-padded)
        gen_tokens     = outputs[j][total_input_len:]
        raw_prediction = tokenizer_eval.decode(gen_tokens, skip_special_tokens=True).strip()
        raw_prediction = raw_prediction.split("\n")[0].strip()

        # Post-processing
        cleaned_prediction = clean_prediction(raw_prediction)
        span_prediction    = find_best_span(cleaned_prediction, sample["context"])

        # Tính metric
        em_raw,  f1_raw  = get_best_score(cleaned_prediction, sample["answers"])
        em_span, f1_span = get_best_score(span_prediction,    sample["answers"])

        all_em_raw.append(em_raw);   all_f1_raw.append(f1_raw)
        all_em_span.append(em_span); all_f1_span.append(f1_span)
        all_predictions.append({
            "id":               sample["id"],
            "question":         sample["question"],
            "ground_truth":     sample["answers"][0]["text"],
            "raw_prediction":   raw_prediction,
            "clean_prediction": cleaned_prediction,
            "span_prediction":  span_prediction,
            "em_raw":  em_raw,  "f1_raw":  f1_raw,
            "em_span": em_span, "f1_span": f1_span,
        })

        idx = len(all_predictions)

        # Log mỗi câu - format giống file 1.5B
        tqdm.write(
            f"[{idx}/{TOTAL}] "
            f"Q: {sample['question'][:40]}... | "
            f"Truth: {sample['answers'][0]['text'][:30]} | "
            f"Raw: {cleaned_prediction[:30]} | "
            f"Span: {span_prediction[:30]} | "
            f"EM_raw={em_raw} EM_span={em_span} "
            f"F1_raw={f1_raw:.3f} F1_span={f1_span:.3f}"
        )

    # Cập nhật tqdm postfix sau mỗi batch
    idx     = len(all_predictions)
    elapsed = time.time() - start_time
    eta     = (elapsed / idx) * (TOTAL - idx) if idx > 0 else 0
    cur_em  = sum(all_em_span) / idx * 100
    cur_f1  = sum(all_f1_span) / idx * 100
    pbar.set_postfix({"EM": f"{cur_em:.1f}%", "F1": f"{cur_f1:.1f}%", "ETA": f"{eta/60:.1f}m"})

    # Tóm tắt mỗi SUMMARY_EVERY câu
    if idx % SUMMARY_EVERY < BATCH_SIZE and idx >= SUMMARY_EVERY:
        tqdm.write(f"\n{'='*60}")
        tqdm.write(f"  [Checkpoint ~{idx}/{TOTAL}] Đã chạy {elapsed:.0f}s | ETA ~{eta/60:.1f} phút")
        tqdm.write(f"  EM  (raw):  {sum(all_em_raw)/idx*100:.2f}%  |  EM  (span): {cur_em:.2f}%")
        tqdm.write(f"  F1  (raw):  {sum(all_f1_raw)/idx*100:.2f}%  |  F1  (span): {cur_f1:.2f}%")
        tqdm.write(f"{'='*60}\n")

total_time = time.time() - start_time
print(f"\nHoàn tất inference! Tổng thời gian: {total_time/60:.1f} phút")
print(f"Tốc độ trung bình: {TOTAL/total_time:.1f} câu/giây")





Bắt đầu đánh giá trên 1062 câu hỏi...
Batch size = 8 | Số batch = 133 | use_cache=True (đã merge)



/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in 

[1/1062] Q: Lò được làm nóng trước khi nướng bao lâu... | Truth: 10 phút | Raw: 10 phút | Span: 10 phút | EM_raw=1 EM_span=1 F1_raw=1.000 F1_span=1.000
[2/1062] Q: Đầu bếp sử dụng gì để trang trí bánh?... | Truth: mấy cái hoa hồi | Raw: hoa hồi | Span: hoa hồi | EM_raw=0 EM_span=0 F1_raw=0.667 F1_span=0.667
[3/1062] Q: Bột ca cao được hòa tan bằng gì?... | Truth: nước ấm | Raw: đường ấm | Span: đường | EM_raw=0 EM_span=0 F1_raw=0.500 F1_span=0.000
[4/1062] Q: Việc làm nóng lò trước khi nướng để làm ... | Truth: tạo rễ tre cho bánh và để bánh | Raw: tạo rèn cho bánh và để bánh đư | Span: cho bánh và để bánh được chín  | EM_raw=0 EM_span=0 F1_raw=0.857 F1_span=0.842
[5/1062] Q: Nếu để hỗn hợp ca cao sôi mạnh thì sẽ mấ... | Truth: nước cốt dừa | Raw: nước cốt dừa | Span: nước cốt dừa | EM_raw=1 EM_span=1 F1_raw=1.000 F1_span=1.000
[6/1062] Q: Nếu sử dụng trứng vịt thì bánh flan sẽ n... | Truth: ngon hơn | Raw: không ảnh hưởng đến chất lượng | Span: không ảnh hưởng đến chất lượng | EM_raw=

In [14]:
# ==========================================
# IN KẾT QUẢ ĐÁNH GIÁ
# ==========================================
total = len(all_em_raw)

em_raw_score   = sum(all_em_raw)  / total * 100
f1_raw_score   = sum(all_f1_raw)  / total * 100
em_span_score  = sum(all_em_span) / total * 100
f1_span_score  = sum(all_f1_span) / total * 100

print("\n" + "=" * 60)
print("   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA")
print("   Model: Qwen2.5-3B-Instruct + DoRA (Unsloth)")
print("=" * 60)
print(f"  Tổng số câu hỏi test: {total}")
print()
print(f"  {'Metric':<25} {'Raw (no post-proc)':>20} {'Best-Span':>20}")
print(f"  {'-'*65}")
print(f"  {'Exact Match (EM)':<25} {em_raw_score:>19.2f}% {em_span_score:>19.2f}%")
print(f"  {'F1-score (token)':<25} {f1_raw_score:>19.2f}% {f1_span_score:>19.2f}%")
print("=" * 60)

# Phân phối F1 (span version)
f1_buckets = {"0-0.2": 0, "0.2-0.5": 0, "0.5-0.8": 0, "0.8-1.0": 0, "1.0": 0}
for f1 in all_f1_span:
    if f1 == 1.0:    f1_buckets["1.0"] += 1
    elif f1 >= 0.8:  f1_buckets["0.8-1.0"] += 1
    elif f1 >= 0.5:  f1_buckets["0.5-0.8"] += 1
    elif f1 >= 0.2:  f1_buckets["0.2-0.5"] += 1
    else:            f1_buckets["0-0.2"] += 1

print(f"\n  EM đúng (span): {sum(all_em_span)}/{total} câu")
print("\n  Phân phối F1-score (Best-Span):")
for bucket, count in f1_buckets.items():
    print(f"    F1 = {bucket}: {count} câu ({count/total*100:.1f}%)")

# Lưu kết quả
with open("eval_results_vlogqa_3b_dora.json", "w", encoding="utf-8") as f:
    json.dump({
        "em_raw":    round(em_raw_score,  4),
        "f1_raw":    round(f1_raw_score,  4),
        "em_span":   round(em_span_score, 4),
        "f1_span":   round(f1_span_score, 4),
        "total":     total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa_3b_dora.json")


   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA
   Model: Qwen2.5-3B-Instruct + DoRA (Unsloth)
  Tổng số câu hỏi test: 1062

  Metric                      Raw (no post-proc)            Best-Span
  -----------------------------------------------------------------
  Exact Match (EM)                        28.72%               30.23%
  F1-score (token)                        49.99%               50.52%

  EM đúng (span): 321/1062 câu

  Phân phối F1-score (Best-Span):
    F1 = 0-0.2: 382 câu (36.0%)
    F1 = 0.2-0.5: 121 câu (11.4%)
    F1 = 0.5-0.8: 165 câu (15.5%)
    F1 = 0.8-1.0: 72 câu (6.8%)
    F1 = 1.0: 322 câu (30.3%)

Kết quả chi tiết đã được lưu tại: eval_results_vlogqa_3b_dora.json


In [ ]:
# ==========================================
# XEM CÁC MẪU SAI ĐỂ PHÂN TÍCH
# ==========================================
wrong_preds = [p for p in all_predictions if p["em"] == 0]
print(f"\nSố câu EM = 0 (sai hoàn toàn): {len(wrong_preds)}")
print("\n--- 5 mẫu sai đầu tiên ---")
for p in wrong_preds[:5]:
    print(f"\n  Q: {p['question']}")
    print(f"  Ground truth: {p['ground_truth']}")
    print(f"  Prediction  : {p['prediction']}")
    print(f"  F1          : {p['f1']:.4f}")

# Lưu kết quả ra file JSON để phân tích sau
with open("eval_results_vlogqa_3b.json", "w", encoding="utf-8") as f:
    json.dump({
        "exact_match": round(exact_match_score, 4),
        "f1_score": round(f1_score_avg, 4),
        "total": total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa_3b.json")